# Selenium을 이용한 기상청 날씨 크롤링

In [5]:
%pip install selenium pandas

Note: you may need to restart the kernel to use updated packages.


In [70]:
# 봇 처럼 여겨지지 않기 위해 주피터 노트북 ipynb 파일 생성
# 크롤링은 어떻게 사이트에서 사람이 하는 것처럼 보일까가 중요

# pip install selenium
from selenium import webdriver

driver = webdriver.Chrome()
driver.set_window_size(1920, 1080)

# URL='https://www.naver.com/'
URL='https://www.weather.go.kr/w/weather/forecast/short-term.do'
driver.get(url=URL)

In [71]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [72]:
# --- 방법 1: 링크 텍스트 사용 (가장 간단하고 추천) ---
# "1시간 간격"이라는 텍스트를 가진 링크를 직접 찾습니다.
print("방법 1: 링크 텍스트로 클릭 시도...")
# WebDriverWait를 사용하여 요소가 클릭 가능할 때까지 최대 10초간 기다립니다.
one_hour_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "1시간 간격"))
)
one_hour_button.click()
print("'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)")

방법 1: 링크 텍스트로 클릭 시도...
'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)


In [73]:
# --- 방법 2: CSS 선택자 사용 ---
print("방법 2: CSS 선택자(클래스)로 클릭 시도...")
table_view_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "a.view-table"))
)
table_view_button.click()
print("'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)")

방법 2: CSS 선택자(클래스)로 클릭 시도...
'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)


In [74]:
from selenium.common.exceptions import NoSuchElementException
import re # 정규표현식 사용 (데이터 정제용)

In [ ]:
# --- 데이터 저장을 위한 빈 리스트 초기화 ---
times = []
weathers = []
temperatures = []
feels_like_temps = []
precip_amounts = []
precip_intensities = []
precip_probabilities = []
wind_directions = []
wind_speeds = []
humidities = []
heatwave_impacts = []

# --- 데이터 추출 로직 ---
try:
    # 데이터 항목들을 포함하는 부모 div 찾기
    # daily_div = driver.find_element(By.CSS_SELECTOR, "div.daily")
    # item_wrap = daily_div.find_element(By.CSS_SELECTOR, "div.item-wrap")
    # 위 코드를 > 를 이용해 한줄로 작성 가능
    item_wrap = driver.find_element(By.CSS_SELECTOR, "div.daily > div.item-wrap")

    # print(item_wrap.get_attribute('outerHTML')) # item_wrap 내용 확인

    # 각 시간대별 데이터 묶음 (ul 태그) 찾기
    item_list = item_wrap.find_elements(By.CSS_SELECTOR, "ul.item")

    print(f"총 {len(item_list)}개의 시간대 데이터를 찾았습니다.")

    # 각 시간대별로 반복 처리
    for item_ul in item_list:
        # 각 ul 내의 li 요소들을 리스트로 가져오기
        # IndexError를 방지하기 위해 li 개수를 먼저 확인하는 것이 더 안전할 수 있습니다.
        try:
             li_elements = item_ul.find_elements(By.TAG_NAME, "li")
             # 최소 필요한 li 개수(예: 10개) 확인 로직 추가 가능
             # if len(li_elements) < 10: continue # 또는 None 추가 후 다음 item으로
        except NoSuchElementException:
             print("경고: 현재 시간대(ul.item)에서 li 요소들을 찾을 수 없습니다. 건너<0xEB><0x8D>니다.")
             # 모든 리스트에 None 추가하고 다음 item_ul로 넘어감
             lists_to_update = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]
             for lst in lists_to_update:
                 lst.append(None)
             continue # 다음 시간대로

        # 각 항목 추출 및 정제 (clean_value 함수 로직 인라인)

        # 1. 시각
        cleaned_time = None
        try:
            time_text = li_elements[0].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            # 정제 로직 (기본 전처리)
            time_text = time_text.strip().replace('&nbsp;', '')
            if time_text and time_text != '-':
                cleaned_time = time_text # 시각은 특별한 숫자 변환 없음
        except (NoSuchElementException, IndexError) as e:
             print(f"시각 처리 오류: {e}") # 디버깅용 로그
        times.append(cleaned_time)


        # 2. 날씨
        cleaned_weather = None
        try:
            # 먼저 wic 클래스 시도
            try:
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span.wic").text
            except NoSuchElementException:
                 # wic 없으면 다른 span 시도
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            
            # 정제 로직 (기본 전처리)
            weather_text = weather_text.strip().replace('&nbsp;', '')
            if weather_text and weather_text != '-':
                 cleaned_weather = weather_text # 날씨는 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"날씨 처리 오류: {e}")
        weathers.append(cleaned_weather)


        # 3. 기온 
        # 참고: 원래 코드에서는 li_elements[2] (3번째 li)에서 추출했으나, 
        # 이전 논의에서 실제 기온은 4번째 li에서 가져오는 것이 맞다고 판단했습니다.
        # 만약 3번째 li의 텍스트에서 첫 숫자를 기온으로 사용하려면 아래 로직 사용
        cleaned_temp = None
        try:
             # 3번째 li의 전체 텍스트 (예: "16℃(16℃)") 에서 첫 숫자 추출
             temp_text_combined = li_elements[2].find_element(By.CSS_SELECTOR, "span.hid.feel").text
             temp_text_combined = temp_text_combined.strip().replace('&nbsp;', '')
             if temp_text_combined and temp_text_combined != '-':
                 match = re.search(r'-?\d+', temp_text_combined) # 첫 번째 숫자 검색
                 if match:
                     cleaned_temp = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"기온 처리 오류: {e}")
        temperatures.append(cleaned_temp)
        

        # 4. 기온 체감 (3번째 li의 span.chill 텍스트)
        cleaned_feels_like = None
        try:
            chill_text = li_elements[2].find_element(By.CSS_SELECTOR, "span.chill").text # 예: (16℃)
            chill_text = chill_text.strip().replace('&nbsp;', '')
            if chill_text and chill_text != '-':
                match = re.search(r'-?\d+', chill_text) # 괄호 안 숫자 검색
                if match:
                    cleaned_feels_like = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"체감기온 처리 오류: {e}")
        feels_like_temps.append(cleaned_feels_like)


        # 5. 강수량
        cleaned_precip_amount = None
        try:
            pcp_text = li_elements[4].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            pcp_text = pcp_text.strip().replace('&nbsp;', '')
            if pcp_text and pcp_text != '-':
                if '빗방울' in pcp_text:
                    cleaned_precip_amount = 0.0 # '빗방울'은 0.0으로 처리
                else:
                    match = re.search(r'\d+\.?\d*', pcp_text) # 소수점 포함 숫자 검색
                    if match:
                        cleaned_precip_amount = float(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수량 처리 오류: {e}")
        precip_amounts.append(cleaned_precip_amount)


        # 6. 강수강도
        cleaned_intensity = None
        try:
            intensity_element = li_elements[5]
            intensity_text = None
            # 먼저 span 찾기 시도
            try:
                intensity_text = intensity_element.find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            except NoSuchElementException:
                 # span 없으면 li 전체 텍스트에서 hid 제외
                 full_text = intensity_element.text
                 hidden_text = ""
                 try:
                     hidden_text = intensity_element.find_element(By.CSS_SELECTOR, "span.hid").text
                 except NoSuchElementException: pass
                 intensity_text = full_text.replace(hidden_text, "").strip()

            # 정제 로직 (기본 전처리)
            intensity_text = intensity_text.strip().replace('&nbsp;', '')
            if intensity_text and intensity_text != '-':
                cleaned_intensity = intensity_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
             print(f"강수강도 처리 오류: {e}")
        precip_intensities.append(cleaned_intensity)


        # 7. 강수확률
        cleaned_prob = None
        try:
            prob_text = li_elements[6].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            prob_text = prob_text.strip().replace('&nbsp;', '')
            if prob_text and prob_text != '-':
                match = re.search(r'\d+', prob_text) # % 제거 후 숫자만
                if match:
                    cleaned_prob = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수확률 처리 오류: {e}")
        precip_probabilities.append(cleaned_prob)


        # 8. 바람 (방향, 속도 분리)
        cleaned_wind_dir = None
        cleaned_wind_spd = None
        try:
            wind_li = li_elements[7]
            # 바람 방향
            try:
                wind_dir_text = wind_li.find_element(By.CSS_SELECTOR, "span.wdic").text
                wind_dir_text = wind_dir_text.strip().replace('&nbsp;', '')
                if wind_dir_text and wind_dir_text != '-':
                     cleaned_wind_dir = wind_dir_text
            except NoSuchElementException: pass # 없으면 None 유지
            # 바람 속도
            try:
                wind_spd_text = wind_li.find_element(By.CSS_SELECTOR, "span.wspd:not(.qwsd)").text
                wind_spd_text = wind_spd_text.strip().replace('&nbsp;', '')
                if wind_spd_text and wind_spd_text != '-':
                    match = re.search(r'\d+', wind_spd_text) # m/s 제거 후 숫자만
                    if match:
                        cleaned_wind_spd = int(match.group(0))
            except NoSuchElementException: pass # 없으면 None 유지
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"바람 처리 오류: {e}")
        wind_directions.append(cleaned_wind_dir)
        wind_speeds.append(cleaned_wind_spd)


        # 9. 습도
        cleaned_humidity = None
        try:
            hum_text = li_elements[8].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            hum_text = hum_text.strip().replace('&nbsp;', '')
            if hum_text and hum_text != '-':
                match = re.search(r'\d+', hum_text) # % 제거 후 숫자만
                if match:
                    cleaned_humidity = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"습도 처리 오류: {e}")
        humidities.append(cleaned_humidity)


        # 10. 폭염 영향
        cleaned_heatwave = None
        try:
            heat_text = li_elements[9].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            heat_text = heat_text.strip().replace('&nbsp;', '')
            if heat_text and heat_text != '-':
                cleaned_heatwave = heat_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"폭염영향 처리 오류: {e}")
        heatwave_impacts.append(cleaned_heatwave)

    # --- 최종 결과 출력 ---
    # (이전과 동일)
    print("\n--- 추출 완료된 리스트 ---")
    print(f"시각: {times}")
    print(f"날씨: {weathers}")
    print(f"기온(℃): {temperatures}")
    print(f"체감기온(℃): {feels_like_temps}")
    print(f"강수량(mm): {precip_amounts}")
    print(f"강수강도: {precip_intensities}")
    print(f"강수확률(%): {precip_probabilities}")
    print(f"바람방향: {wind_directions}")
    print(f"바람속도(m/s): {wind_speeds}")
    print(f"습도(%): {humidities}")
    print(f"폭염영향: {heatwave_impacts}")

except NoSuchElementException as e:
    print(f"오류: 필수 요소를 찾을 수 없습니다. CSS 선택자를 확인하세요. ({e})")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")
    import traceback
    traceback.print_exc()

# finally:
#     # 작업 완료 후 드라이버 종료
#     # driver.quit()

<div class="item-wrap">
                        
                        
                        
                        
                        
                            <ul class="item vs-item  v-item-first " data-date="2025-04-22" data-time="20:00">
                                <li><span class="hid">시각: </span><span>20시</span></li>
                                <li><span class="hid">날씨: </span><span class="wic DB05-Q1" title="약한비">약한비</span></li>
                                <li><span class="hid">기온(체감온도) </span><span class="hid feel">16℃<span class="chill">(16℃)</span></span></li>
                                <li><span class="hid">체감온도: </span><span>16℃</span></li>
                                <li class="pcp "><span class="hid">강수량: </span><span>~1<span class="unit">mm</span></span></li>
                                <li>
                                
                                	
	                                
	                                	<span

In [66]:
keys = ['시각', '날씨', '기온(℃)', '체감기온(℃)', '강수량(mm)', '강수강도', '강수확률(%)', '바람방향', '바람속도(m/s)', '습도(%)', '폭염영향']
# 제공된 리스트 변수들을 사용한다고 가정 (times, weathers, temperatures 등)
list_of_lists = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]

structured_data = []
num_items = len(times) # 모든 리스트 길이가 같다고 가정

for i in range(num_items):
    record = {}
    for j, key in enumerate(keys):
         # list_of_lists[j][i] 를 사용하여 올바른 값에 접근
         record[key] = list_of_lists[j][i] 
    structured_data.append(record)

# 이제 structured_data를 JSON으로 변환하여 API에 전달할 수 있습니다.
import json
json_data = json.dumps(structured_data, ensure_ascii=False, indent=2) 
print(type(json_data))
print(json_data)

<class 'str'>
[
  {
    "시각": "19시",
    "날씨": "약한비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 2.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "북풍",
    "바람속도(m/s)": 1,
    "습도(%)": 95,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "약한비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 1.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 95,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "구름 많음",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북풍",
    "바람속도(m/s)": 3,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨": "구름 많음",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북풍",
    "바람속도(m/s)": 3,
    "습도(%)": 85,
    "폭염영향": null
  },
  {
    "시각": "23시",
    "날씨": "구름 많음",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,

# GEMINI API 연동

In [36]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY 환경 변수를 설정해주세요.")

In [ ]:
from google import genai

client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain how AI works in a few words",
)

print(response.text)

AI learns patterns from data to make predictions or decisions.



In [41]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
[
  {
    "시각": "18시",
    "날씨": "보통비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 3.0,
    "강수강도": "보통비",
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 7,
    "습도(%)": 95,
    "폭염영향": null
  },
  {
    "시각": "19시",
    "날씨": "약한비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 2.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 14,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "빗방울",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": 0.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 11,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "흐림",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 13,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨": "구름

In [47]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

## 시간대별 날씨 요약 및 변화 분석

**전반적인 날씨 흐름:**

18시부터 0시까지의 날씨는 비에서 흐림으로 점차 변화하는 추세를 보입니다. 기온은 16℃에서 15℃로 소폭 하락했으며, 습도는 95%에서 85%로 감소했습니다. 폭염의 영향은 관측되지 않았습니다.

**시간대별 상세 분석 및 주목할 만한 변화:**

*   **18시:** 보통비가 내리고 있으며, 습도가 95%로 매우 높습니다. 바람은 남서풍이 7m/s로 불고 있습니다.
*   **19시:** 강수량이 줄어 약한비로 바뀌었습니다. 바람 방향이 서풍으로 바뀌고 풍속이 14m/s로 강해진 것이 주목할 만합니다.
*   **20시:** 빗방울이 떨어지면서 강수량이 0mm가 되었습니다. 그러나 강수강도는 여전히 약한비로 표시되어 있습니다. 바람은 남서풍으로 다시 바뀌었고, 풍속은 11m/s로 줄었습니다.
*   **21시:** 비가 그치고 흐린 날씨로 바뀌었습니다. 강수량 및 강수강도 정보가 null로 표시됩니다. 바람은 남서풍이 13m/s로 불고 있습니다.
*   **22시:** 구름이 많아졌으며, 날씨는 흐림에서 구름 많음으로 변화했습니다. 다른 요소들은 큰 변화가 없습니다.
*   **23시:** 날씨는 다시 흐림으로 돌아왔습니다. 바람 방향이 북풍으로 바뀌고 풍속이 4m/s로 크게 약해진 것이 특징입니다. 강수확률은 30%로, 비가 다시 올 가능성이 존재합니다.
*   **0시:** 날씨는 흐림이 유지되고, 바람 방향은 북풍, 풍속은 4m/s로 23시와 동일합니다. 습도가 85%로 소폭 감소했습니다.

**주요 변화 요약:**

*   **강수:** 18시 '보통비'에서 21시 '흐림'으로, 강수가 점차적으로 종료되었습니다.
*   **풍속 및 풍향:** 19시에 서풍으로 바뀌면서 풍속이 급격히 증가(7m/s -> 14m/s)했다가, 이후 남서풍으로 돌아오면서 풍속이 감소했습니다. 23시에는 북풍으로 바뀌면서 풍속이 다시 크게 감소했습니다.
*   **습도:** 전반적으로 습도가 감소하는

In [51]:
import datetime
current_time_local = datetime.datetime.now()
formatted_time= current_time_local.strftime("%Y-%m-%d %H:%M:%S")
formatted_time

'2025-04-22 18:53:47'

In [52]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** 2025-04-22 18:53:47

**날씨 데이터:**
```json
[
  {
    "시각": "18시",
    "날씨": "보통비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 3.0,
    "강수강도": "보통비",
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 7,
    "습도(%)": 95,
    "폭염영향": null
  },
  {
    "시각": "19시",
    "날씨": "약한비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 2.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 14,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "빗방울",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": 0.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 11,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "흐림",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 13,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨"

In [53]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

현재 시간은 18시 53분입니다.

제공된 날씨 데이터에 따르면 18시에는 "보통비"가 내리고 있고, 강수량은 3.0mm입니다. 19시에는 "약한비"가 내립니다. 20시에는 "빗방울"이 내립니다.

따라서 지금 외출하신다면 **우산이 필요합니다.**



In [54]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** 2025-04-22 18:53:47

**날씨 데이터:**
```json
[
  {
    "시각": "18시",
    "날씨": "보통비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 3.0,
    "강수강도": "보통비",
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 7,
    "습도(%)": 95,
    "폭염영향": null
  },
  {
    "시각": "19시",
    "날씨": "약한비",
    "기온(℃)": 16,
    "체감기온(℃)": 16,
    "강수량(mm)": 2.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 14,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "빗방울",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": 0.0,
    "강수강도": "약한비",
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 11,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "흐림",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 13,
    "습도(%)": 90,
    "폭염영향": null
  },
  {
    "시각": "22

In [55]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

현재 시각은 2025년 4월 22일 18시 53분입니다. 날씨 데이터를 분석한 결과, 지금 외출한다면 다음과 같은 드레스 코디를 추천합니다.

**필수:**

*   **겉옷:** 16℃의 기온과 바람을 고려하여 얇은 겉옷 (가디건, 바람막이, 얇은 자켓 등)을 챙기세요. 특히 바람이 강하게 불 수 있으므로 바람을 막아주는 기능성 소재의 겉옷이 좋습니다.
*   **우산:** 현재 "보통비"가 내리고 있고, 이후에도 비가 오락가락할 수 있으므로 우산을 꼭 챙기세요. 접는 우산이 휴대하기 편리합니다.
*   **신발:** 비에 젖어도 괜찮은 신발을 선택하세요. 방수 기능이 있는 운동화나 레인부츠도 좋은 선택입니다.

**추가 고려 사항:**

*   **상의:** 16℃는 긴팔 티셔츠나 얇은 니트 정도가 적당합니다. 습도가 높으므로 통기성이 좋은 소재를 선택하는 것이 좋습니다.
*   **하의:** 긴 바지나 무릎 아래 길이의 스커트/원피스를 추천합니다.
*   **액세서리:** 스카프나 머플러를 챙기면 바람으로부터 목을 보호하고 스타일을 더할 수 있습니다.

**요약:**

비가 오고 바람이 부는 날씨이므로, 따뜻하게 입고 비에 대비하는 것이 중요합니다.

*   **얇은 겉옷 (바람막이, 가디건)**
*   **긴팔 상의**
*   **긴 바지 또는 무릎 아래 스커트/원피스**
*   **우산**
*   **방수 신발**
*   **(선택) 스카프/머플러**

